# ⚖️ pack-quantized × sm75

asymmetric 4bit / group_size 128 / marlin 可否。


In [ ]:
import glob
import pathlib
import subprocess
import sys
import zipfile

subprocess.run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
                "--no-deps", "vllm==0.32.0", "-d", "/tmp/w"],
               capture_output=True, text=True)
whl = glob.glob("/tmp/w/*.whl")
print("wheel:", whl)
if not whl:
    raise SystemExit("wheel 取得失敗")
zipfile.ZipFile(whl[0]).extractall("/tmp/vl")
root = pathlib.Path("/tmp/vl/vllm")
print("py files:", len(list(root.rglob("*.py"))))

import json
import os

ct = root / "model_executor/layers/quantization/compressed_tensors"
print("=== compressed_tensors ディレクトリ ===")
print("exists:", ct.exists())
if ct.exists():
    for f in sorted(ct.rglob("*.py")):
        print("  ", f.relative_to(ct), f.stat().st_size)

qinit = root / "model_executor/layers/quantization/__init__.py"
q = qinit.read_text(errors="replace")
i = q.find("QuantizationMethods = Literal[")
print("\n=== QuantizationMethods 全文 ===")
print(q[i:i + 1400] if i > 0 else "not found")
print("compressed-tensors を含む:", "compressed-tensors" in q)

print("\n=== METADATA の compressed-tensors 依存 ===")
meta_name = [n for n in __import__("zipfile").ZipFile(
    "/tmp/w/" + os.listdir("/tmp/w")[0]).namelist()
    if n.endswith(".dist-info/METADATA")][0]
z = __import__("zipfile").ZipFile("/tmp/w/" + os.listdir("/tmp/w")[0])
meta = z.read(meta_name).decode("utf-8", "replace")
for line in meta.splitlines():
    if "compressed" in line.lower() or "marlin" in line.lower():
        print("  ", line[:160])

print("\n=== scheme クラス ===")
for f in (sorted(ct.rglob("*scheme*.py")) if ct.exists() else []):
    t = f.read_text(errors="replace")
    for i2, l in enumerate(t.splitlines(), 1):
        s = l.strip()
        if s.startswith("class ") or "def can_implement" in s or "def __init__" in s:
            print(f"  {f.name}:{i2}: {s[:140]}")

print("\n=== symmetric / asymmetric / group_size の扱い ===")
for f in sorted(root.rglob("*.py")):
    p = str(f)
    if not any(k in p for k in ("compressed_tensors", "marlin", "awq")):
        continue
    t = f.read_text(errors="replace")
    for i2, l in enumerate(t.splitlines(), 1):
        s = l.strip()
        if ("symmetric" in s or "asymmetric" in s) and (
                "raise" in s or "assert" in s or "support" in s or "if " in s):
            print(f"  {f.relative_to(root)}:{i2}: {s[:150]}")

print("\n=== 4bit / capability ガード（marlin・cutlass）===")
for f in sorted(root.rglob("*.py")):
    p = str(f)
    if not any(k in p for k in ("marlin", "cutlass", "quantization/utils")):
        continue
    t = f.read_text(errors="replace")
    for i2, l in enumerate(t.splitlines(), 1):
        s = l.strip()
        if "capability" in s and ("<" in s or ">=" in s or "==" in s):
            print(f"  {f.relative_to(root)}:{i2}: {s[:150]}")

print("\n=== 実際の config と突き合わせ ===")

def find_root():
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    # slug 一致を最優先。無ければ qwen を含むもの、最後に全候補。
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]


ROOT = find_root()

cfg = json.load(open(os.path.join(ROOT, "config.json")))
qc = cfg.get("quantization_config") or {}
for k in ("quant_method", "format", "version", "kv_cache_scheme"):
    print(f"  {k}: {qc.get(k)}")
grp = (qc.get("config_groups") or {}).get("group_0") or {}
print("  group_0 targets:", grp.get("targets"))
print("  weights:", json.dumps(grp.get("weights"), ensure_ascii=False))
print("  ignore count:", len(qc.get("ignore") or []))
